# Per-cell-type exercise rescue analysis (pseudobulk DESeq2) — trimmed for the manuscript

## What this notebook produces
| Manuscript panel | Cell | Output |
|---|---|---|
| Fig. 2b (rescued genes per cell type) | §4 → §6 | `rescue_summary_per_celltype_DESeq2.csv`, `F_MAIN_vizA1_DESeq2_log_Reds.pdf` |
| Fig. 2c (Hallmark NES, disease vs restoration) | §3, §5 | `pathway_NES_per_celltype_DESeq2.csv`, `F_MAIN_pathway_restoration_heatmap_DESeq2.pdf` |
| not shown in manuscript (robustness record) | §7 | `F_SUPP_threshold_2x2_DESeq2.pdf` |

The same two main PDFs are also rendered by `make_reha_rescue_figures.py` from the CSVs above.

## Method in one paragraph
- **DE**: pseudobulk (raw counts summed per sample × cell type) → PyDESeq2, design `~type`, Wald test, **raw (unshrunken) log2FC**.
- **Thresholds**: BH-adjusted p (padj) < 0.10 **and** |log2FC| > 0.25, applied per cell type × contrast. These are looser than Ma et al. 2020 (padj < 0.05, |log2FC| > 0.5); §7 shows both.
- **Contrasts**: SED vs Ctrl (MI DEGs) and Ex vs SED (Ex DEGs).
- **Classes (Ma et al. *Cell* 2020)**: rescue = MI DEG ∩ Ex DEG, opposite direction; side-effect = same direction; Ex-specific = Ex DEG that is not an MI DEG; MI-only = MI DEG with no significant Ex change.
- **Cell-type filter**: ≥ 50 nuclei in total and ≥ 3 nuclei in every group; cell types with zero MI DEGs are dropped from figures.

**Refs**: Ma S et al. *Cell* 2020 (PMID 32109414); Squair JW et al. *Nat Commun* 2021; Heumos L et al. *Nat Rev Genet* 2023.


In [1]:
import warnings
warnings.simplefilter('ignore', category=Warning)
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import decoupler as dc
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import gseapy as gp
from scipy.cluster.hierarchy import linkage, leaves_list
from matplotlib.colors import TwoSlopeNorm
from matplotlib.gridspec import GridSpec
import matplotlib as mpl

sc.settings.verbosity = 1
sc.set_figure_params(dpi=100, facecolor='white', fontsize=12)
plt.rcParams['axes.grid'] = False

OUT = Path('./outputs/restoration')
FIG = OUT / 'figures'
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(exist_ok=True)
RNG_SEED = 0
print('scanpy', sc.__version__, '| anndata', ad.__version__, '| decoupler', dc.__version__, '| gseapy', gp.__version__)

ModuleNotFoundError: No module named 'decoupler'

## 1. Load data

In [2]:
adata = sc.read_h5ad('./adata/allpopulation_curated_Ctrl_6W.h5ad')
print(adata)
print('obs cols:', adata.obs.columns.tolist())
print('layers:', list(adata.layers.keys()))
print('raw is None:', adata.raw is None)

AnnData object with n_obs × n_vars = 42168 × 2000
    obs: 'sample', 'type', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'cell_type_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'scrublet', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
obs cols: ['sample', 'type', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type']
layers: []
raw is None: False


## 2. Labels, thresholds (padj < 0.10, |log2FC| > 0.25) and cell-type filter

In [3]:
CELLTYPE_COL = 'cell_type'
SAMPLE_COL   = 'sample'

type_map = {'Ctrl':'Ctrl','Sham':'Ctrl','SED6W':'SED','SED':'SED','Reha6W':'Ex','Ex':'Ex','Reha':'Ex'}
adata.obs['type'] = adata.obs['type'].astype(str).map(type_map)
adata.obs['type'] = pd.Categorical(adata.obs['type'], categories=['Ctrl','SED','Ex'], ordered=True)
print(adata.obs['type'].value_counts())

# Ma 2020 spec: ≥50 cells total AND ≥3 cells per type-group
DEG_LFC_CUT = 0.25
DEG_P_CUT = 0.10
THR_LABEL = f'FDR<{DEG_P_CUT}, |log2FC|>{DEG_LFC_CUT}'
print('Threshold:', THR_LABEL)

# Class color/label maps (Ma 2020 framework)
CLASS_COLORS_MA = {'mi_only':'#999999','rescue':'#2ca25f','side_effect':'#d9534f','ex_specific':'#4a90d9'}
CLASS_LABELS_MA = {'mi_only':'MI-only','rescue':'Rescue DEG','side_effect':'Side-effect DEG','ex_specific':'Ex-specific DEG'}
mi_classes = ['mi_only','rescue','side_effect']

MIN_CELLS_TOTAL = 50
MIN_CELLS_PER_GROUP = 3
ct_x_sample = pd.crosstab(adata.obs[CELLTYPE_COL], adata.obs[SAMPLE_COL])
ct_x_type = adata.obs.groupby([CELLTYPE_COL,'type'], observed=True).size().unstack(fill_value=0)
totals = ct_x_sample.sum(axis=1)
ok_ct = []
for ct in totals.index:
    if totals[ct] < MIN_CELLS_TOTAL: continue
    if ct not in ct_x_type.index: continue
    if (ct_x_type.loc[ct] < MIN_CELLS_PER_GROUP).any(): continue
    ok_ct.append(ct)
excluded_ct = [c for c in ct_x_sample.index if c not in ok_ct]
print(f'\nMAIN celltypes (n={len(ok_ct)}):', ok_ct)
print(f'EXCLUDED (n={len(excluded_ct)}):', excluded_ct)
print('cells per (celltype × sample):'); display(ct_x_sample)
print('cells per (celltype × type):'); display(ct_x_type)

type
Ex      17206
SED     15216
Ctrl     9746
Name: count, dtype: int64
Threshold: FDR<0.1, |log2FC|>0.25

MAIN celltypes (n=11): ['B cells', 'Cardiomyocytes', 'Endocardial Cells', 'Endothelial Cells', 'Epicardial Cells', 'Fibroblasts', 'Lympatic Endothelial Cells', 'Lymphoid Cells', 'Myeloid Cells', 'Pericytes', 'Smooth Muscle Cells']
EXCLUDED (n=0): []
cells per (celltype × sample):


sample,Ctrl1,Ctrl2,Reha6W1,Reha6W2,SED6W1,SED6W2
cell_type,,,,,,
B cells,11,11,57,79,11,42
Cardiomyocytes,1971,2033,2027,2120,838,2561
Endocardial Cells,229,205,201,721,118,516
Endothelial Cells,987,834,1963,2891,1451,3659
Epicardial Cells,4,7,51,259,51,44
Fibroblasts,898,912,1671,1792,643,2342
Lympatic Endothelial Cells,26,32,123,109,98,172
Lymphoid Cells,8,12,26,61,21,20
Myeloid Cells,210,255,720,1011,347,1089


cells per (celltype × type):


type,Ctrl,SED,Ex
cell_type,,,
B cells,22,53,136
Cardiomyocytes,4004,3399,4147
Endocardial Cells,434,634,922
Endothelial Cells,1821,5110,4854
Epicardial Cells,11,95,310
Fibroblasts,1810,2985,3463
Lympatic Endothelial Cells,58,270,232
Lymphoid Cells,20,41,87
Myeloid Cells,465,1436,1731


## Class definitions (quoted from Ma et al. 2020, STAR Methods)

> 'rescue DEGs' were defined as the upregulated or downregulated genes among the aging DEGs that were downregulated or upregulated, respectively, upon CR; 'side-effect DEGs' were defined as the upregulated or downregulated genes among the aging DEGs that were further upregulated or downregulated, respectively, upon CR; 'CR-specific DEGs' were defined as the upregulated or downregulated genes among the CR DEGs that were not markedly changed during aging.
> — Ma et al. *Cell* 2020, STAR Methods

## 3. Pseudobulk DESeq2 (primary differential expression)

Counts are taken from `adata.raw.X`. Genes need ≥ 10 counts in ≥ 4 pseudobulk samples. Classification uses the raw Wald log2FC; the apeglm-shrunken value is stored only as an extra column for SED vs Ctrl.

In [4]:
adata_pb_src = ad.AnnData(X=adata.raw.X.copy().astype(np.float32),
                          obs=adata.obs.copy(), var=adata.raw.var.copy())
adata_pb_src = adata_pb_src[adata_pb_src.obs[CELLTYPE_COL].isin(ok_ct)].copy()
pdata = dc.pp.pseudobulk(adata_pb_src, sample_col=SAMPLE_COL, groups_col=CELLTYPE_COL,
                          mode='sum', skip_checks=False, verbose=False)
pdata.obs['type'] = pdata.obs['type'].astype(str)
pdata.obs['type'] = pd.Categorical(pdata.obs['type'], categories=['Ctrl','SED','Ex'], ordered=True)
print('pseudobulk shape:', pdata.shape)
pdata.write_h5ad(OUT / 'pseudobulk_counts.h5ad')

MIN_COUNT = 10; MIN_SAMPLES_EXPR = 4
def filter_lowexpr(pd_ct, min_count=MIN_COUNT, min_samples=MIN_SAMPLES_EXPR):
    X = pd_ct.X.toarray() if hasattr(pd_ct.X,'toarray') else pd_ct.X
    keep = (X >= min_count).sum(axis=0) >= min_samples
    return pd_ct[:, keep].copy()

def run_deseq_one(pd_ct):
    """Single-fit DESeq2 (ref=Ctrl) for both contrasts.
    Uses RAW Wald LFC (no apeglm shrinkage) for symmetric classification matching Wilcoxon
    (apeglm over-shrinks Ex_vs_SED when most genes have small effect — see Codex review).
    Shrunken LFC is computed and stored as a separate column for visualization only.
    """
    pd_ct = filter_lowexpr(pd_ct)
    if pd_ct.n_vars < 100 or pd_ct.obs['type'].nunique() < 3: return None
    pd_ct.obs['type'] = pd_ct.obs['type'].astype(str)
    dds = DeseqDataSet(adata=pd_ct, design='~type', ref_level=('type','Ctrl'),
                       n_cpus=1, refit_cooks=True, quiet=True)
    dds.deseq2()
    out = {}
    for ref, alt, name in [('Ctrl','SED','SED_vs_Ctrl'),('SED','Ex','Ex_vs_SED')]:
        ds = DeseqStats(dds, contrast=['type', alt, ref], n_cpus=1, quiet=True)
        ds.summary()
        df = ds.results_df.copy()
        df['log2FoldChange_raw'] = df['log2FoldChange']  # primary for classification
        # Optional: compute shrunken LFC for visualization (only available for ref=Ctrl coefficients)
        shrunk = None
        if name == 'SED_vs_Ctrl':
            try:
                ds2 = DeseqStats(dds, contrast=['type','SED','Ctrl'], n_cpus=1, quiet=True)
                ds2.summary(); ds2.lfc_shrink(coeff='type[T.SED]')
                shrunk = ds2.results_df['log2FoldChange'].copy()
            except Exception: pass
        df['log2FoldChange_shrunk'] = shrunk if shrunk is not None else float('nan')
        df['shrink_applied_for_viz'] = shrunk is not None
        out[name] = df
    return out

results = {}
for ct in ok_ct:
    pd_ct = pdata[pdata.obs[CELLTYPE_COL] == ct].copy()
    if pd_ct.obs['type'].nunique() < 3 or pd_ct.n_obs < 4: continue
    res = run_deseq_one(pd_ct)
    if res is None: continue
    results[ct] = res
    for contrast, df in res.items():
        df.to_csv(OUT / f'{ct}_pseudobulk_{contrast}.csv')
print(f'pseudobulk DESeq2 completed for {len(results)} celltypes')

pseudobulk shape: (66, 24225)
pseudobulk DESeq2 completed for 11 celltypes


## 4. Rescue classification (→ Fig. 2b input)

Applies the class definitions above to the DESeq2 results (raw log2FC, padj). Writes `rescue_classification_all_DESeq2.csv` and `rescue_summary_per_celltype_DESeq2.csv`.

In [6]:
def classify_genes_deseq(deseq_res, lfc_cut=DEG_LFC_CUT, p_cut=DEG_P_CUT):
    mi = deseq_res['SED_vs_Ctrl'].rename(columns={'log2FoldChange':'log2FC_MI','padj':'padj_MI'})[['log2FC_MI','padj_MI']]
    ex = deseq_res['Ex_vs_SED'].rename(columns={'log2FoldChange':'log2FC_Ex','padj':'padj_Ex'})[['log2FC_Ex','padj_Ex']]
    df = mi.join(ex, how='outer')
    is_mi_deg = (df['padj_MI'] < p_cut) & (df['log2FC_MI'].abs() > lfc_cut)
    is_ex_deg = (df['padj_Ex'] < p_cut) & (df['log2FC_Ex'].abs() > lfc_cut)
    sign_mi = np.sign(df['log2FC_MI']); sign_ex = np.sign(df['log2FC_Ex'])
    cls = pd.Series('not_significant', index=df.index, dtype=object)
    rescue = is_mi_deg & is_ex_deg & (sign_mi * sign_ex < 0)
    side_eff = is_mi_deg & is_ex_deg & (sign_mi * sign_ex > 0)
    ex_specific = is_ex_deg & (~is_mi_deg)
    mi_only = is_mi_deg & (~is_ex_deg)
    cls[mi_only]='mi_only'; cls[rescue]='rescue'; cls[side_eff]='side_effect'; cls[ex_specific]='ex_specific'
    df['class'] = cls
    df['is_mi_deg'] = is_mi_deg.fillna(False)
    df['is_ex_deg'] = is_ex_deg.fillna(False)
    return df

class_results_d = {}; all_rows_d = []
for ct, res in results.items():
    df = classify_genes_deseq(res)
    df['celltype'] = ct; df['gene'] = df.index
    class_results_d[ct] = df
    all_rows_d.append(df.reset_index(drop=True))
rest_df_d = pd.concat(all_rows_d, ignore_index=True)
rest_df_d.to_csv(OUT / 'rescue_classification_all_DESeq2.csv', index=False)

class_tab_d = pd.crosstab(rest_df_d.celltype, rest_df_d['class'])
mi_n_d = rest_df_d.groupby('celltype').apply(lambda d: d.is_mi_deg.sum())
summary_d = pd.DataFrame({
    'n_MI_DEGs':      mi_n_d,
    'n_rescue':       class_tab_d.get('rescue', 0),
    'n_side_effect':  class_tab_d.get('side_effect', 0),
    'n_mi_only':      class_tab_d.get('mi_only', 0),
    'n_ex_specific':  class_tab_d.get('ex_specific', 0),
    'rescue_rate':    (class_tab_d.get('rescue', 0) / mi_n_d).round(3),
    'side_effect_rate': (class_tab_d.get('side_effect', 0) / mi_n_d).round(3),
    'mi_only_rate':   (class_tab_d.get('mi_only', 0) / mi_n_d).round(3),
}).fillna(0)
summary_d.to_csv(OUT / 'rescue_summary_per_celltype_DESeq2.csv')
print('=== DESeq2-based Ma 2020 classification summary ===')
display(summary_d)

# === Filter: drop celltypes with zero DEG (e.g., B cells, Lymphoid with n_MI_DEGs==0) ===
NONEMPTY_CT = summary_d[summary_d['n_MI_DEGs'] > 0].index.tolist()
EMPTY_CT = [c for c in summary_d.index if c not in NONEMPTY_CT]
print(f'Non-empty celltypes (kept): {NONEMPTY_CT}')
print(f'Empty celltypes (dropped from figures): {EMPTY_CT}')
# Restrict downstream containers
class_results_d = {ct: df for ct, df in class_results_d.items() if ct in NONEMPTY_CT}
results = {ct: r for ct, r in results.items() if ct in NONEMPTY_CT}
summary_d = summary_d.loc[NONEMPTY_CT]


=== DESeq2-based Ma 2020 classification summary ===


,n_MI_DEGs,n_rescue,n_side_effect,n_mi_only,n_ex_specific,rescue_rate,side_effect_rate,mi_only_rate
celltype,,,,,,,,
B cells,0,0,0,0,0,0.000,0.000,0.000
Cardiomyocytes,4141,351,4,3786,46,0.085,0.001,0.914
Endocardial Cells,416,3,0,413,4,0.007,0.000,0.993
Endothelial Cells,997,2,0,995,0,0.002,0.000,0.998
Epicardial Cells,139,7,1,131,15,0.050,0.007,0.942
Fibroblasts,1270,18,0,1252,11,0.014,0.000,0.986
Lympatic Endothelial Cells,135,0,0,135,0,0.000,0.000,1.000
Lymphoid Cells,0,0,0,0,0,0.000,0.000,0.000
Myeloid Cells,369,6,1,362,5,0.016,0.003,0.981


Non-empty celltypes (kept): ['Cardiomyocytes', 'Endocardial Cells', 'Endothelial Cells', 'Epicardial Cells', 'Fibroblasts', 'Lympatic Endothelial Cells', 'Myeloid Cells', 'Pericytes', 'Smooth Muscle Cells']
Empty celltypes (dropped from figures): ['B cells', 'Lymphoid Cells']


## 5a. Pathway prep — pinned Hallmark GMT + mouse→human ortholog table

In [5]:
# === Pathway analysis: pinned Hallmark GMT + mygene ortholog mapping (Codex review fix #4) ===
HALLMARK_GMT = str(OUT / 'genesets' / 'MSigDB_Hallmark_2020_pinned.gmt')
ortho_path = OUT / 'genesets' / 'mouse_human_ortholog_mygene.tsv'
ortho_df = pd.read_csv(ortho_path, sep='\t')
M2H = dict(zip(ortho_df['mouse'], ortho_df['human']))
print(f'Pinned Hallmark GMT: {HALLMARK_GMT}')
print(f'Ortholog table: {ortho_path}, mouse→human pairs={len(M2H)}')

def mouse_to_human_rank(df, lfc_col='log2FC', tie_col=None):
    """Convert mouse-symbol DataFrame to human-symbol Series for GSEA prerank.
    Drop zero LFC + tie-break + ortholog mapping (lossy uppercase fallback for unmapped)."""
    d = df.dropna(subset=[lfc_col]).copy()
    d = d[d[lfc_col] != 0]  # drop zero-LFC genes (mostly dropouts)
    d = d.reset_index().rename(columns={'index':'gene'})
    if 'gene' not in d.columns:
        d.columns = ['gene'] + list(d.columns[1:])
    d['human'] = d['gene'].map(M2H).fillna(d['gene'].str.upper())  # mygene first, uppercase fallback
    agg = {lfc_col: 'mean'}
    if tie_col and tie_col in d.columns: agg[tie_col] = 'mean'
    d = d.groupby('human').agg(agg).reset_index()
    if tie_col and tie_col in d.columns:
        d = d.sort_values([lfc_col, tie_col], ascending=[False, False])
    else:
        d = d.sort_values(lfc_col, ascending=False)
    return pd.Series(d[lfc_col].values, index=d['human'].values)

# Sanity check: how many of our genes map to Hallmark gene set
hallmark_genes_set = set()
with open(HALLMARK_GMT) as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) > 2:
            hallmark_genes_set.update(parts[2:])
naive_in_hk = sum(1 for g in adata.raw.var_names if g.upper() in hallmark_genes_set)
mapped_in_hk = sum(1 for g in adata.raw.var_names if M2H.get(g, g.upper()) in hallmark_genes_set)
print(f'Genes in Hallmark: naive uppercase={naive_in_hk}, mygene ortholog={mapped_in_hk}')


Pinned Hallmark GMT: outputs/restoration/genesets/MSigDB_Hallmark_2020_pinned.gmt
Ortholog table: outputs/restoration/genesets/mouse_human_ortholog_mygene.tsv, mouse→human pairs=15206
Genes in Hallmark: naive uppercase=4052, mygene ortholog=4176


## 5b. Hallmark GSEA and heatmap (→ Fig. 2c)

`gseapy.prerank` on DESeq2 log2FC ranks (Wald statistic as tie-break, zero-log2FC genes dropped), 100 permutations. `*` = FDR q < 0.10; black border = NES sign flips between contrasts with |NES| > 1 in both.

In [ ]:
# Match the Viz A1 bar graph ordering (n_rescue desc) - skips B cells/Lymphoid at the front
_rows = []
for ct, df in class_results_d.items():
    n_mi = int(df['is_mi_deg'].sum())
    n_r = int((df['class']=='rescue').sum())
    _rows.append({'celltype':ct, 'n_rescue':n_r})
ct_order = pd.DataFrame(_rows).set_index('celltype').sort_values('n_rescue', ascending=False).index.tolist()
PATHWAY_ROWS_D = []
for ct, res in results.items():
    for contrast, name in [('SED_vs_Ctrl','SED_vs_Ctrl'),('Ex_vs_SED','Ex_vs_SED')]:
        s = mouse_to_human_rank(res[contrast].rename(columns={'log2FoldChange':'log2FC','stat':'score'}),
                                  lfc_col='log2FC', tie_col='score')
        try:
            pre = gp.prerank(rnk=s, gene_sets=HALLMARK_GMT, threads=1, min_size=10, max_size=500,
                              permutation_num=100, outdir=None, seed=RNG_SEED, verbose=False)
            r = pre.res2d.copy(); r['celltype']=ct; r['contrast']=name
            PATHWAY_ROWS_D.append(r)
        except Exception as e:
            print(f'prerank failed {ct} {name}: {e}')

pw_d = pd.concat(PATHWAY_ROWS_D, ignore_index=True) if PATHWAY_ROWS_D else pd.DataFrame()
for c in ['NES','ES','NOM p-val','FDR q-val','FWER p-val']:
    if c in pw_d.columns: pw_d[c] = pd.to_numeric(pw_d[c], errors='coerce')
pw_d = pw_d[pw_d['celltype'].isin(NONEMPTY_CT)].copy()
pw_d.to_csv(OUT / 'pathway_NES_per_celltype_DESeq2.csv', index=False)
nes_wide_d = pw_d.pivot_table(index=['celltype','Term'], columns='contrast', values='NES').dropna()
nes_wide_d['flip'] = (np.sign(nes_wide_d['SED_vs_Ctrl']) != np.sign(nes_wide_d['Ex_vs_SED'])) & \
                     (nes_wide_d['SED_vs_Ctrl'].abs() > 1) & (nes_wide_d['Ex_vs_SED'].abs() > 1)
print(f'DESeq2 pathway results: {pw_d.shape}, sign-flip & |NES|>1: {nes_wide_d.flip.sum()}')

ct_order_d = [c for c in ct_order if c in results.keys()]
fdr_d_des = pw_d[pw_d.contrast=='SED_vs_Ctrl'].pivot_table(index='Term', columns='celltype', values='FDR q-val')
fdr_r_des = pw_d[pw_d.contrast=='Ex_vs_SED'].pivot_table(index='Term', columns='celltype', values='FDR q-val')
sig_in_either_d = ((fdr_d_des < 0.10).any(axis=1)) | ((fdr_r_des < 0.10).any(axis=1))
flip_terms_set_d = set(nes_wide_d.reset_index().query('flip').Term.unique().tolist())
sig_terms_set_d = set(sig_in_either_d[sig_in_either_d].index.tolist())
pathway_rows_d = sorted(flip_terms_set_d & sig_terms_set_d)
print(f'DESeq2 sign-flip: {len(flip_terms_set_d)} | sig FDR<0.10 either: {len(sig_terms_set_d)} | both = main rows: {len(pathway_rows_d)}')
print(f'DESeq2 main pathway rows: {len(pathway_rows_d)}')

if len(pathway_rows_d) > 0 and len(ct_order_d) > 0:
    nes_d2 = pw_d[pw_d.contrast=='SED_vs_Ctrl'].pivot_table(index='Term', columns='celltype', values='NES').reindex(index=pathway_rows_d, columns=ct_order_d)
    nes_r2 = pw_d[pw_d.contrast=='Ex_vs_SED'].pivot_table(index='Term', columns='celltype', values='NES').reindex(index=pathway_rows_d, columns=ct_order_d)
    pd_d2 = pw_d[pw_d.contrast=='SED_vs_Ctrl'].pivot_table(index='Term', columns='celltype', values='FDR q-val').reindex(index=pathway_rows_d, columns=ct_order_d)
    pd_r2 = pw_d[pw_d.contrast=='Ex_vs_SED'].pivot_table(index='Term', columns='celltype', values='FDR q-val').reindex(index=pathway_rows_d, columns=ct_order_d)

    if len(pathway_rows_d) > 1:
        Z = linkage(nes_d2.fillna(0).values, method='average', metric='euclidean')
        oi = leaves_list(Z)
        pathway_rows_d = [pathway_rows_d[i] for i in oi]
        nes_d2 = nes_d2.loc[pathway_rows_d]; nes_r2 = nes_r2.loc[pathway_rows_d]
        pd_d2 = pd_d2.loc[pathway_rows_d]; pd_r2 = pd_r2.loc[pathway_rows_d]

    abs_max = max(np.nanmax(np.abs(nes_d2.values)), np.nanmax(np.abs(nes_r2.values)), 2.0)
    norm = TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
    fig = plt.figure(figsize=(2.5 + 0.55*len(ct_order_d)*2, max(6, 0.32*len(pathway_rows_d))))
    gs = GridSpec(1, 3, width_ratios=[len(ct_order_d), len(ct_order_d), 0.8], wspace=0.4)
    ax_d = fig.add_subplot(gs[0,0]); ax_r = fig.add_subplot(gs[0,1]); ax_cb = fig.add_subplot(gs[0,2])

    sns.heatmap(nes_d2, ax=ax_d, cmap='RdBu_r', norm=norm, cbar=False, linewidths=0.4, linecolor='white', mask=nes_d2.isna())
    for i, term in enumerate(pathway_rows_d):
        for j, ct in enumerate(ct_order_d):
            p = pd_d2.iloc[i,j]
            if pd.notna(p) and p < 0.10:
                ax_d.text(j+0.5, i+0.5, '*', ha='center', va='center', fontsize=10, color='black')
    ax_d.set_title('SED vs Ctrl  (disease)', fontsize=12); ax_d.set_xlabel(''); ax_d.set_ylabel('Hallmark pathway', fontsize=10)
    ax_d.set_xticklabels(ax_d.get_xticklabels(), rotation=45, ha='right')

    sns.heatmap(nes_r2, ax=ax_r, cmap='RdBu_r', norm=norm, cbar=False, linewidths=0.4, linecolor='white', mask=nes_r2.isna())
    for i, term in enumerate(pathway_rows_d):
        for j, ct in enumerate(ct_order_d):
            p = pd_r2.iloc[i,j]
            if pd.notna(p) and p < 0.10:
                ax_r.text(j+0.5, i+0.5, '*', ha='center', va='center', fontsize=10, color='black')
            d = nes_d2.iloc[i,j]; r = nes_r2.iloc[i,j]
            if pd.notna(d) and pd.notna(r) and (np.sign(d) != np.sign(r)) and abs(d) > 1 and abs(r) > 1:
                ax_r.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor='black', lw=1.5))
    ax_r.set_title('Ex vs SED  (restoration)', fontsize=12); ax_r.set_xlabel(''); ax_r.set_ylabel(''); ax_r.set_yticklabels([])
    ax_r.set_xticklabels(ax_r.get_xticklabels(), rotation=45, ha='right')
    mpl.colorbar.ColorbarBase(ax_cb, cmap=mpl.cm.RdBu_r, norm=norm, orientation='vertical', label='NES')
    plt.suptitle('MAIN · Pathway restoration (pseudobulk DESeq2 LFC ranks, Hallmark GSEA, local GMT + mygene ortholog)\n* = FDR q<0.10 ; black border = sign-flip & |NES|>1 (rescue)',
                 fontsize=11, y=1.005)
    plt.savefig(FIG / 'F_MAIN_pathway_restoration_heatmap_DESeq2.pdf', bbox_inches='tight')
    plt.show()
    print(f'DESeq2 main heatmap saved: {len(pathway_rows_d)} pathways × {len(ct_order_d)} celltypes')

2026-05-05 20:02:29,105 [WARNING] Duplicated values found in preranked stats: 0.01% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
2026-05-05 20:02:29,316 [WARNING] Duplicated values found in preranked stats: 0.01% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


## 6. Rescued genes per cell type (→ Fig. 2b)

In [ ]:
# === MAIN · Viz A1 (DESeq2 pseudobulk, log10 count, Reds colormap) ===
# DESeq2-only — Wilcoxon removed per Heumos 2023 best-practice (pseudobulk for between-condition DE)
import matplotlib as mpl

rows = []
for ct, df in class_results_d.items():
    n_mi = int(df['is_mi_deg'].sum())
    n_r = int((df['class']=='rescue').sum())
    rows.append({'celltype':ct, 'n_MI':n_mi, 'n_rescue':n_r,
                 'rescue_pct': n_r/n_mi*100 if n_mi>0 else 0})
m_d = pd.DataFrame(rows).set_index('celltype')
order_D = m_d.sort_values('n_rescue', ascending=False).index.tolist()

# DESeq2-specific PCT_MAX (rescaled now that Wilcoxon is gone)
PCT_MAX = max(m_d['rescue_pct'].max(), 5)
PCT_MAX = float(np.ceil(PCT_MAX/2.5)*2.5)  # round up to 2.5%
norm_pct = mpl.colors.Normalize(vmin=0, vmax=PCT_MAX)
cmap_pct = plt.cm.Reds
print(f'DESeq2-only PCT_MAX for colorbar: {PCT_MAX:.1f}%')

fig, ax = plt.subplots(figsize=(10, max(4, 0.5*len(order_D))))
sub = m_d.reindex(order_D)
counts = sub['n_rescue'].fillna(0).values.astype(float)
pcts   = sub['rescue_pct'].fillna(0).values
n_mis  = sub['n_MI'].fillna(0).astype(int).values
colors = [cmap_pct(norm_pct(p)) for p in pcts]
x_log  = np.log10(counts + 1)
max_x  = max(x_log.max(), 1)
ax.barh(range(len(sub)), np.maximum(x_log, max_x*0.005), color=colors, edgecolor='black', linewidth=0.6)
for i, ct in enumerate(sub.index):
    ax.text(max_x*1.02, i,
            f'{int(counts[i]):>4d}  |  {pcts[i]:>5.1f}%  (of {int(n_mis[i])} MI DEGs)',
            va='center', fontsize=9, family='monospace')
ax.set_xticks([np.log10(t+1) for t in [1,10,100,1000]])
ax.set_xticklabels(['1','10','100','1000'])
ax.set_xlim(0, max_x*1.6)
ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub.index, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('# rescue genes (log10 scale)')
ax.set_title(f'MAIN · pseudobulk DESeq2 (Heumos 2023 standard, {THR_LABEL})', loc='left', fontsize=11)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
sm_ = mpl.cm.ScalarMappable(cmap=cmap_pct, norm=norm_pct)
fig.colorbar(sm_, ax=ax, location='right', shrink=0.7, label='rescue rate %')
plt.tight_layout()
plt.savefig(FIG / 'F_MAIN_vizA1_DESeq2_log_Reds.pdf', bbox_inches='tight')
plt.show()
print('Saved F_MAIN_vizA1_DESeq2_log_Reds.pdf')


## 7. Threshold sensitivity (not shown in manuscript; kept as a robustness record)

DESeq2 only. Strict (padj < 0.05, |log2FC| > 0.5; Ma 2020) vs relaxed (padj < 0.10, |log2FC| > 0.25; used in Fig. 2b), as counts and as percentages.

In [ ]:
# === SUPP · DESeq2 threshold sensitivity (strict vs relaxed × counts/%) ===
def classify_simple(mi, ex, lfc_cut, p_cut, lfc_col_mi, p_col_mi, lfc_col_ex, p_col_ex):
    df = mi[[lfc_col_mi, p_col_mi]].rename(columns={lfc_col_mi:'lfc_mi', p_col_mi:'padj_mi'}).join(
         ex[[lfc_col_ex, p_col_ex]].rename(columns={lfc_col_ex:'lfc_ex', p_col_ex:'padj_ex'}), how='outer')
    is_mi = (df['padj_mi'] < p_cut) & (df['lfc_mi'].abs() > lfc_cut)
    is_ex = (df['padj_ex'] < p_cut) & (df['lfc_ex'].abs() > lfc_cut)
    sm = np.sign(df['lfc_mi']); se = np.sign(df['lfc_ex'])
    cls = pd.Series('not_significant', index=df.index, dtype=object)
    cls[is_mi & ~is_ex] = 'mi_only'
    cls[is_mi & is_ex & (sm * se < 0)] = 'rescue'
    cls[is_mi & is_ex & (sm * se > 0)] = 'side_effect'
    cls[~is_mi & is_ex] = 'ex_specific'
    return cls, is_mi.fillna(False)

CLASS_COLS = {'mi_only':'#999999','rescue':'#2ca25f','side_effect':'#d9534f'}
CLASS_LABELS = {'mi_only':'MI-only','rescue':'Rescue','side_effect':'Side-effect'}
mi_classes = ['mi_only','rescue','side_effect']
configs = [(0.5, 0.05, 'strict (FDR<0.05, |LFC|>0.5)'),
           (0.25, 0.10, 'relaxed (FDR<0.10, |LFC|>0.25)')]
ALL_CT = list(adata.obs[CELLTYPE_COL].astype(str).unique())

def get_summary(lfc, p):
    rows = []
    for ct in ALL_CT:
        mp = OUT/f'{ct}_pseudobulk_SED_vs_Ctrl.csv'
        ep = OUT/f'{ct}_pseudobulk_Ex_vs_SED.csv'
        if not (mp.exists() and ep.exists()): continue
        mi = pd.read_csv(mp, index_col=0); ex = pd.read_csv(ep, index_col=0)
        cls, is_mi = classify_simple(mi, ex, lfc, p, 'log2FoldChange','padj','log2FoldChange','padj')
        rows.append({'celltype':ct, 'n_MI':int(is_mi.sum()),
                     'rescue':int((cls=='rescue').sum()),
                     'side_effect':int((cls=='side_effect').sum()),
                     'mi_only':int((cls=='mi_only').sum())})
    return pd.DataFrame(rows).set_index('celltype')

ALL_CT_ord = sorted(ALL_CT, key=lambda c: m_d.loc[c,'n_rescue'] if c in m_d.index else 0, reverse=True)

fig = plt.figure(figsize=(15, 9))
gs = GridSpec(2, 2, hspace=0.45, wspace=0.30)
for idx, (lfc, p, ttl) in enumerate(configs):
    df = get_summary(lfc, p).reindex(ALL_CT_ord).fillna(0)
    df['n_MI'] = df['n_MI'].astype(int)
    cnt = df.sort_values('n_MI', ascending=False)
    cts = cnt.index.tolist()
    tab = pd.DataFrame({cl: df.loc[cts, cl].astype(int).values for cl in mi_classes}, index=cts)
    tab_pct = tab.div(tab.sum(axis=1).replace(0, np.nan), axis=0).fillna(0) * 100
    # counts panel
    ax_c = fig.add_subplot(gs[idx, 0])
    mxv = tab.values.max() if tab.values.max()>0 else 1
    left = np.zeros(len(cts))
    for cl in mi_classes:
        v = tab[cl].values
        ax_c.barh(cts, v, left=left, color=CLASS_COLS[cl], label=CLASS_LABELS[cl] if (idx==0) else None,
                  edgecolor='white', linewidth=0.4)
        for i, vv in enumerate(v):
            if vv > mxv*0.05: ax_c.text(left[i]+vv/2, i, str(int(vv)), ha='center', va='center', fontsize=7, color='white')
        left += v
    for i, t in enumerate(tab.sum(axis=1).values):
        if t>0: ax_c.text(t+mxv*0.01, i, str(int(t)), va='center', fontsize=7)
    ax_c.set_xlabel('# MI DEGs'); ax_c.set_title(f'DESeq2 · {ttl} · counts', loc='left', fontsize=10)
    ax_c.invert_yaxis(); ax_c.tick_params(axis='y', labelsize=8)
    ax_c.spines['top'].set_visible(False); ax_c.spines['right'].set_visible(False)
    if idx==0: ax_c.legend(loc='lower right', frameon=False, fontsize=8)
    # pct panel
    ax_p = fig.add_subplot(gs[idx, 1])
    left = np.zeros(len(cts))
    for cl in mi_classes:
        v = tab_pct[cl].values
        ax_p.barh(cts, v, left=left, color=CLASS_COLS[cl], edgecolor='white', linewidth=0.4)
        for i, vv in enumerate(v):
            if vv > 5: ax_p.text(left[i]+vv/2, i, f'{vv:.0f}', ha='center', va='center', fontsize=7, color='white')
        left += v
    ax_p.set_xlabel('% of MI DEGs'); ax_p.set_xlim(0,100)
    ax_p.set_title(f'DESeq2 · {ttl} · %', loc='left', fontsize=10)
    ax_p.invert_yaxis(); ax_p.set_yticklabels([])
    ax_p.spines['top'].set_visible(False); ax_p.spines['right'].set_visible(False)
plt.suptitle('Threshold sensitivity (DESeq2 only, Ma 2020 framework, strict vs relaxed × counts/%)', fontsize=12, y=0.995)
plt.savefig(FIG / 'F_SUPP_threshold_2x2_DESeq2.pdf', bbox_inches='tight')
plt.show()
print('Saved F_SUPP_threshold_2x2_DESeq2.pdf')
